# Dispatch On ACTIVSg

Companion notebook for Tutorial 12. Run a 24-hour `ACTIVSg2000` dispatch study, compare sequential versus time-coupled SCED, and plot a nodal-price heat map from the sequential LMPs.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import pandas as pd
import surge

from workflow_common import find_activsg_case, find_activsg_time_series_root

case_path = find_activsg_case("2000")
time_series_root = find_activsg_time_series_root()

if case_path is None or time_series_root is None:
    print("Populate examples/cases/case_ACTIVSg2000/ and research/test-cases/data/ACTIVSg_Time_Series/ before running this notebook.")
else:
    print("Case:", case_path)
    print("Time series:", time_series_root)


In [ ]:
if case_path is not None and time_series_root is not None:
    net = surge.load(case_path)
    activsg = surge.dispatch.read_tamu_activsg_time_series(net, time_series_root, case="2000")
    net = activsg.network_with_nameplate_overrides(net)
    bus_df = net.bus_dataframe().reset_index()
    branch_df = net.branch_dataframe().reset_index()
    print("Imported periods:", activsg.periods)
    print("Buses with coordinates:", int(bus_df[["latitude", "longitude"]].notna().all(axis=1).sum()))
    activsg.report


In [ ]:
def make_request(imported, periods, coupling):
    return {
        "formulation": "dc",
        "coupling": coupling,
        "commitment": "all_committed",
        "timeline": imported.timeline(periods),
        "profiles": imported.dc_dispatch_profiles(periods),
        "market": {
            "generator_cost_modeling": {
                "use_pwl_costs": True,
                "pwl_cost_breakpoints": 20,
            }
        },
        "network": {
            "enforce_thermal_limits": True,
            "enforce_flowgates": False,
            "use_loss_factors": True,
        },
    }


if case_path is not None and time_series_root is not None:
    seq_24 = surge.solve_dispatch(
        net,
        make_request(activsg, 24, "period_by_period"),
        lp_solver="highs",
    )
    tc_24 = surge.solve_dispatch(
        net,
        make_request(activsg, 24, "time_coupled"),
        lp_solver="highs",
    )
    pd.DataFrame(
        [
            {
                "study": "Sequential",
                "coupling": seq_24.study["coupling"],
                "periods": seq_24.study["periods"],
                "total_cost": seq_24.summary["total_cost"],
            },
            {
                "study": "Time-coupled",
                "coupling": tc_24.study["coupling"],
                "periods": tc_24.study["periods"],
                "total_cost": tc_24.summary["total_cost"],
            },
        ]
    )


In [ ]:
coords = (
    bus_df[["bus_id", "latitude", "longitude"]]
    .dropna(subset=["latitude", "longitude"])
    .drop_duplicates(subset=["bus_id"])
    .set_index("bus_id")
)


def period_total_withdrawals(result):
    return [
        sum(bus["withdrawals_mw"] for bus in period["bus_results"])
        for period in result.periods
    ]


def lmp_frame(result, period_index):
    return (
        bus_df.merge(
            pd.DataFrame(result.periods[period_index]["bus_results"]),
            left_on="bus_id",
            right_on="bus_number",
            how="inner",
        )
        .dropna(subset=["latitude", "longitude"])
        .copy()
    )


def branch_segments():
    segments = []
    for row in branch_df.itertuples(index=False):
        if row.from_bus not in coords.index or row.to_bus not in coords.index:
            continue
        from_pt = coords.loc[row.from_bus]
        to_pt = coords.loc[row.to_bus]
        segments.append(
            [
                (from_pt["longitude"], from_pt["latitude"]),
                (to_pt["longitude"], to_pt["latitude"]),
            ]
        )
    return segments


def plot_lmp_map(frame, title, ax):
    lines = LineCollection(branch_segments(), colors="#d8d8d8", linewidths=0.3, zorder=1)
    ax.add_collection(lines)
    scatter = ax.scatter(
        frame["longitude"],
        frame["latitude"],
        c=frame["lmp"],
        cmap="viridis",
        s=18,
        edgecolors="none",
        zorder=2,
    )
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    return scatter


In [ ]:
if case_path is not None and time_series_root is not None:
    peak_hour = int(np.argmax(period_total_withdrawals(seq_24)))
    seq_map = lmp_frame(seq_24, peak_hour)

    fig, ax = plt.subplots(1, 1, figsize=(9, 7), constrained_layout=True)
    scatter = plot_lmp_map(seq_map, f"Sequential SCED, hour {peak_hour}", ax)
    fig.colorbar(scatter, ax=ax, shrink=0.85, label="LMP ($/MWh)")
    plt.show()

    seq_map.sort_values("lmp", ascending=False)[["bus_id", "name", "lmp", "mcc", "mlc", "withdrawals_mw"]].head(10)
